# Data Collection: Thu thập dữ liệu API Chất lượng Không khí (Open-Meteo)

Notebook này thực hiện quy trình tự động hóa việc thu thập dữ liệu từ các nguồn API công khai. Mục tiêu cuối cùng là tạo ra một bộ dữ liệu chứa tọa độ địa lý, nhiệt độ, độ ẩm và nồng độ bụi mịn (PM2.5, PM10) của các tỉnh thành Việt Nam.

## Mục lục
1. [Thiết lập môi trường và Thư viện](#1-thiết-lập-môi-trường-và-thư-viện)
2. [Crawl Tọa độ địa lý (Geocoding API)](#2-crawl-tọa-độ-địa-lý-geocoding-api)
3. [Crawl Chất lượng Không khí & Thời tiết](#3-crawl-chất-lượng-không-khí--thời-tiết)
4. [Tổng hợp dữ liệu và Xuất file (Export)](#4-tổng-hợp-dữ-liệu-và-xuất-file-export)


## 1. Thiết lập môi trường và Thư viện
Import các thư viện cần thiết cho việc gọi API (requests), thao tác dữ liệu dạng bảng (pandas) và xử lý thời gian (time).


In [2]:
import requests
import pandas as pd
import time
import warnings
import os
import json
from IPython.display import display
warnings.filterwarnings('ignore') # Tắt các cảnh báo không cần thiết

print("Đã import thư viện thành công!")


Đã import thư viện thành công!


## 2. Crawl Tọa độ địa lý (Geocoding API)
Để lấy dữ liệu thời tiết cho một địa phương, Open-Meteo yêu cầu phải truyền vào Vĩ độ (`latitude`) và Kinh độ (`longitude`). Do đó, bước đầu tiên là dùng Geocoding API để dịch tên của 63 tỉnh thành ra tọa độ chuẩn.

*(Lưu ý: Để test code nhanh, danh sách dưới đây đang dùng `.head(5)` hoặc cắt `[:5]` để chạy thử 5 tỉnh đầu tiên. Khi chạy thật, bạn chỉ việc bỏ hàm cắt này đi)*


In [ ]:
# Danh sách 63 tỉnh thành Việt Nam
provinces_list = [
    'Hà Nội', 'Hồ Chí Minh', 'Đà Nẵng', 'Hải Phòng', 'Cần Thơ', 'An Giang', 'Bà Rịa - Vũng Tàu', 
    'Bắc Giang', 'Bắc Kạn', 'Bạc Liêu', 'Bắc Ninh', 'Bến Tre', 'Bình Định', 'Bình Dương', 
    'Bình Phước', 'Bình Thuận', 'Cà Mau', 'Cao Bằng', 'Đắk Lắk', 'Đắk Nông', 'Điện Biên', 
    'Đồng Nai', 'Đồng Tháp', 'Gia Lai', 'Hà Giang', 'Hà Nam', 'Hà Tĩnh', 'Hải Dương', 
    'Hậu Giang', 'Hòa Bình', 'Hưng Yên', 'Khánh Hòa', 'Kiên Giang', 'Kon Tum', 'Lai Châu', 
    'Lâm Đồng', 'Lạng Sơn', 'Lào Cai', 'Long An', 'Nam Định', 'Nghệ An', 'Ninh Bình', 
    'Ninh Thuận', 'Phú Thọ', 'Phú Yên', 'Quảng Bình', 'Quảng Nam', 'Quảng Ngãi', 'Quảng Ninh', 
    'Quảng Trị', 'Sóc Trăng', 'Sơn La', 'Tây Ninh', 'Thái Bình', 'Thái Nguyên', 'Thanh Hóa', 
    'Thừa Thiên Huế', 'Tiền Giang', 'Trà Vinh', 'Tuyên Quang', 'Vĩnh Long', 'Vĩnh Phúc', 'Yên Bái'
]

def get_coordinates(province_name):
    """Hàm lấy Vĩ độ, Kinh độ từ tên tỉnh"""
    url = f"https://geocoding-api.open-meteo.com/v1/search?name={province_name}&count=1&language=vi"
    try:
        response = requests.get(url)
        data = response.json()
        if 'results' in data and len(data['results']) > 0:
            result = data['results'][0]
            return {"Province": province_name, "Latitude": result['latitude'], "Longitude": result['longitude']}
    except Exception as e:
        print(f"Lỗi khi tra cứu {province_name}: {e}")
    
    # Trả về giá trị rỗng nếu lỗi
    return {"Province": province_name, "Latitude": None, "Longitude": None}

# 2.1. Vòng lặp thu thập tọa độ (Demo 5 tỉnh)
print("Bắt đầu thu thập tọa độ...")
province_coords = []
for p in provinces_list:  # <-- Đổi thành provinces_list để chạy toàn bộ 63 tỉnh
    coords = get_coordinates(p)
    province_coords.append(coords)
    time.sleep(1)  # Tạm nghỉ 1s để tránh lỗi Rate Limit

# 2.2. Đưa dữ liệu vào DataFrame
df_provinces = pd.DataFrame(province_coords)
display(df_provinces)

os.makedirs('../data', exist_ok=True)
# Lưu danh sách tọa độ vừa crawl được vào file provinces.json
with open('../data/provinces.json', 'w', encoding='utf-8') as f:
    json.dump(province_coords, f, ensure_ascii=False, indent=2)
print("Đã crawl thành công và lưu file tại data/provinces.json")


Bắt đầu thu thập tọa độ...


,Province,Latitude,Longitude
0,Hà Nội,21.02450,105.84117
1,Hồ Chí Minh,10.82302,106.62965
2,Đà Nẵng,16.06778,108.22083
3,Hải Phòng,20.86481,106.68345
4,Cần Thơ,10.03711,105.78825


Đã crawl thành công và lưu file tại data/provinces.json


## 3. Crawl Chất lượng Không khí & Thời tiết
Sử dụng Dataframe tọa độ `df_provinces` vừa thu thập được ở Bước 2. Ta sẽ lặp qua từng dòng, truyền Vĩ độ (`Latitude`) và Kinh độ (`Longitude`) vào 2 API:
1. **Weather Forecast API**: Lấy Nhiệt độ và Độ ẩm.
2. **Air Quality API**: Lấy nồng độ PM10 và PM2.5.


In [4]:
def fetch_weather_and_aqi(lat, lon):
    """Hàm lấy dữ liệu Thời tiết và AQI dựa trên tọa độ"""
    weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m,relative_humidity_2m"
    aqi_url = f"https://air-quality-api.open-meteo.com/v1/air-quality?latitude={lat}&longitude={lon}&current=pm10,pm2_5"
    
    record = {}
    try:
        # Gọi API Thời tiết
        w_res = requests.get(weather_url).json()
        if 'current' in w_res:
            record['Temperature (°C)'] = w_res['current'].get('temperature_2m')
            record['Humidity (%)'] = w_res['current'].get('relative_humidity_2m')
            
        # Gọi API Không khí
        a_res = requests.get(aqi_url).json()
        if 'current' in a_res:
            record['PM2.5 (µg/m³)'] = a_res['current'].get('pm2_5')
            record['PM10 (µg/m³)'] = a_res['current'].get('pm10')
            
    except Exception as e:
        print(f"Lỗi API: {e}")
        
    return record

# 3.1. Thu thập dữ liệu theo từng tỉnh
print("Bắt đầu thu thập dữ liệu AQI & Thời tiết...")
weather_aqi_data = []

for index, row in df_provinces.iterrows():
    if pd.notnull(row['Latitude']):
        print(f"  ➜ Lấy dữ liệu cho: {row['Province']}")
        api_data = fetch_weather_and_aqi(row['Latitude'], row['Longitude'])
        
        # Gắn tên tỉnh vào dict để làm khóa kết nối (key) ở Bước 4
        api_data['Province'] = row['Province'] 
        weather_aqi_data.append(api_data)
        
        time.sleep(1) # Nghỉ 1s tránh Rate Limit

# 3.2. Đưa kết quả vào DataFrame
df_aqi_results = pd.DataFrame(weather_aqi_data)
display(df_aqi_results)


Bắt đầu thu thập dữ liệu AQI & Thời tiết...
  ➜ Lấy dữ liệu cho: Hà Nội
  ➜ Lấy dữ liệu cho: Hồ Chí Minh
  ➜ Lấy dữ liệu cho: Đà Nẵng
  ➜ Lấy dữ liệu cho: Hải Phòng
  ➜ Lấy dữ liệu cho: Cần Thơ


,Temperature (°C),Humidity (%),PM2.5 (µg/m³),PM10 (µg/m³),Province
0,26.8,97,25.8,26.8,Hà Nội
1,25.6,92,49.9,51.6,Hồ Chí Minh
2,26.8,93,19.6,19.9,Đà Nẵng
3,26.4,97,14.3,14.9,Hải Phòng
4,24.9,96,30.3,32.5,Cần Thơ


## 4. Tổng hợp dữ liệu và Xuất file (Export)
Thực hiện phép `Merge` (tương tự như LEFT JOIN trong SQL) để nối Bảng tọa độ và Bảng chất lượng không khí lại thành một Dataset duy nhất. Sau đó, lưu dữ liệu xuống máy dưới định dạng `.csv` để phục vụ cho phân tích EDA ở Notebook tiếp theo.


In [ ]:
# 4.1. Hợp nhất (Merge) 2 Dataframes dựa trên cột 'Province'
df_final = pd.merge(df_provinces, df_aqi_results, on="Province", how="inner")

print("Dữ liệu sau khi tổng hợp:")
display(df_final.head())

# 4.2. Xuất dữ liệu ra file CSV
filename = "../data/vietnam_air_quality_dataset.csv"
df_final.to_csv(filename, index=False, encoding='utf-8-sig')

print(f"\n Quy trình hoàn tất! Đã lưu dữ liệu ra file: {filename}")


Dữ liệu sau khi tổng hợp:


,Province,Latitude,Longitude,Temperature (°C),Humidity (%),PM2.5 (µg/m³),PM10 (µg/m³)
0,Hà Nội,21.02450,105.84117,26.8,97,25.8,26.8
1,Hồ Chí Minh,10.82302,106.62965,25.6,92,49.9,51.6
2,Đà Nẵng,16.06778,108.22083,26.8,93,19.6,19.9
3,Hải Phòng,20.86481,106.68345,26.4,97,14.3,14.9
4,Cần Thơ,10.03711,105.78825,24.9,96,30.3,32.5



 Quy trình hoàn tất! Đã lưu dữ liệu ra file: vietnam_air_quality_dataset.csv
